# Quantization, from scratch

This notebook builds LLM weight quantization from the one formula every method shares, then shows the single fact that makes it hard for LLMs (**outliers**) and the memory it buys. It is the executable companion to the [Quantization](../10-Quantization.md) concept page and is structurally identical to [`quantization.py`](quantization.py).

We will, in order:
1. implement **affine int8** quantization (symmetric and asymmetric) and measure the reconstruction error;
2. inject an **outlier column** and watch per-tensor quantization break while per-group fixes it;
3. quantize a weight block to **int4 group-wise** and compute the exact memory reduction;
4. recompute the **Llama-2-70B** memory headline across precisions.

Every claim is `assert`-ed before it is printed. Runs on CPU in well under a second — no GPU needed.

## Setup: constants and an honest device line

We hoist the quantization constants (bit-widths, the int8/int4 integer maxima, the group size) so there are no magic numbers in the logic. We **detect** the best accelerator but **pin** the reproducible trace to CPU — the quantization math is elementwise `round`/`clamp`, identical on any device, and CPU keeps the printed numbers reproducible. The printed device line says exactly what ran.

In [1]:
import torch

INT8_BITS, INT4_BITS = 8, 4
# Signed symmetric range for b bits is [-(2^(b-1)-1), 2^(b-1)-1]: +/-127 for int8, +/-7 for int4.
INT8_QMAX = (1 << (INT8_BITS - 1)) - 1   # 127
INT4_QMAX = (1 << (INT4_BITS - 1)) - 1   # 7
UINT8_QMIN, UINT8_QMAX = 0, (1 << INT8_BITS) - 1   # [0, 255] for asymmetric
GROUP_SIZE = 64           # one scale per 64 consecutive weights (a common GPTQ/AWQ default)
FP16_BYTES_PER_PARAM = 2  # the baseline precision LLM weights ship in
SEED = 0

DETECTED_DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
DEVICE = "cpu"  # pin: identical math everywhere; CPU keeps the trace reproducible
torch.manual_seed(SEED)
print("device:", f"{DEVICE} (detected {DETECTED_DEVICE}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0


## 1. Affine quantization — the ruler

Quantization maps a real value `x` to the nearest tick on a low-bit ruler: `q = round(x/s) + z`, and back with `x_hat = s*(q - z)`. **Symmetric** fixes `z = 0` (the ruler is centered on zero — right for weights). The scale `s = max|x| / qmax` is set so the loudest value lands exactly on `+/-qmax`. The only lossy step is `round`; the error is bounded by half a step, `s/2`.

In [2]:
def quantize_symmetric_int8(x, qmax=INT8_QMAX):
    """Symmetric affine quantization to signed int8: zero-point fixed at 0, store only the scale."""
    scale = x.abs().max().item() / qmax          # tick spacing: loudest value pins +/-qmax
    if scale == 0.0:                             # guard an all-zero tensor (no range)
        scale = 1.0
    q = torch.round(x / scale)                   # round-to-nearest tick — THE lossy step
    q = torch.clamp(q, -qmax, qmax)              # stay on the int8 grid
    return q.to(torch.int8), scale

def dequantize_symmetric(q, scale):
    """x_hat = s * q  (zero-point is 0)."""
    return q.to(torch.float32) * scale

def reconstruction_error(x, x_hat):
    """Mean absolute reconstruction error |x - x_hat| — the price of quantization, in real units."""
    return (x - x_hat).abs().mean().item()

x = torch.tensor([-1.50, -0.30, 0.00, 0.42, 0.95, 2.10], device=DEVICE)
q_sym, s_sym = quantize_symmetric_int8(x)
x_sym = dequantize_symmetric(q_sym, s_sym)
err_sym = reconstruction_error(x, x_sym)
print(f"x              : {x.tolist()}")
print(f"symmetric  scale s = {s_sym:.6f}   zero-point z = 0 (fixed)")
print(f"  quantized ints : {q_sym.tolist()}")
print(f"  dequantized    : {[round(v, 4) for v in x_sym.tolist()]}")
print(f"  mean|err|      : {err_sym:.6f}")
assert err_sym <= s_sym / 2 + 1e-6, "symmetric error must stay within half a quant step"
print("  (error within the s/2 bound — the loudest value 2.10 pinned q = 127)")

x              : [-1.5, -0.30000001192092896, 0.0, 0.41999998688697815, 0.949999988079071, 2.0999999046325684]
symmetric  scale s = 0.016535   zero-point z = 0 (fixed)
  quantized ints : [-91, -18, 0, 25, 57, 127]
  dequantized    : [-1.5047, -0.2976, 0.0, 0.4134, 0.9425, 2.1]
  mean|err|      : 0.003530
  (error within the s/2 bound — the loudest value 2.10 pinned q = 127)


### Asymmetric: slide the ruler to a lopsided range

When values are **not** centered on zero (a post-GELU activation with a short negative tail and a long positive one), symmetric wastes integer codes on the unused side. **Asymmetric** pairs the scale with an integer zero-point `z`, fitting the actual `[min, max]` window so its step is finer. We compute `z` so `x_min` lands on `qmin` — and `z` is *not* clamped to `[0,255]` (only the codes are).

In [3]:
def quantize_asymmetric_int8(x, qmin=UINT8_QMIN, qmax=UINT8_QMAX):
    """Asymmetric affine quantization to unsigned int8 [0,255]: scale AND integer zero-point."""
    x_min, x_max = x.min().item(), x.max().item()
    scale = (x_max - x_min) / (qmax - qmin)      # map real window [min,max] onto integer [0,255]
    if scale == 0.0:
        scale = 1.0
    zero_point = round(qmin - x_min / scale)     # integer offset so x_min lands on qmin; NOT clamped
    q = torch.round(x / scale) + zero_point      # affine map: scale THEN shift by the zero-point
    q = torch.clamp(q, qmin, qmax)               # clamp the CODES into [0,255] (z itself stays unclamped)
    return q.to(torch.int16), scale, zero_point

def dequantize_asymmetric(q, scale, zero_point):
    """x_hat = s * (q - z)."""
    return (q.to(torch.float32) - zero_point) * scale

# Lopsided: short negative tail, long positive one (the shape of a post-GELU activation).
a = torch.tensor([-0.50, -0.10, 0.30, 1.20, 2.50, 4.00], device=DEVICE)
q_asym, s_asym, z_asym = quantize_asymmetric_int8(a)
err_asym = reconstruction_error(a, dequantize_asymmetric(q_asym, s_asym, z_asym))
q_a_sym, s_a_sym = quantize_symmetric_int8(a)
err_a_sym = reconstruction_error(a, dequantize_symmetric(q_a_sym, s_a_sym))
print(f"lopsided tensor a : {a.tolist()}")
print(f"asymmetric scale s = {s_asym:.6f}   zero-point z = {z_asym}")
print(f"  quantized ints : {q_asym.tolist()}")
print(f"  mean|err| asym : {err_asym:.6f}   (symmetric on same a: {err_a_sym:.6f})")
assert err_asym < err_a_sym, "asymmetric must beat symmetric on a lopsided range"
print("  -> asymmetric < symmetric on a lopsided range (no levels wasted on the unused side)")

lopsided tensor a : [-0.5, -0.10000000149011612, 0.30000001192092896, 1.2000000476837158, 2.5, 4.0]
asymmetric scale s = 0.017647   zero-point z = 28
  quantized ints : [0, 22, 45, 96, 170, 255]
  mean|err| asym : 0.003922   (symmetric on same a: 0.006562)
  -> asymmetric < symmetric on a lopsided range (no levels wasted on the unused side)


## 2. The outlier problem — the crux of LLM quantization

In large LLMs a few **activation channels** become 10–100× louder than the rest, in fixed feature dimensions across nearly every token. On the ruler, one giant value forces a huge scale `s`, crushing every ordinary value. We reproduce it on a weight matrix with one **outlier column** (input channel 17, present in every row) and compare three granularities:

- **per-tensor** — one scale for all of `W` (the loud column sets it);
- **per-channel** — one scale per *row* (does **not** help: every row contains the loud column);
- **per-group** — one scale per group of columns (isolates the outlier to its own group).

In [4]:
def quantize_per_tensor_int8(w):
    """One scale for the WHOLE matrix — the coarsest granularity."""
    q, scale = quantize_symmetric_int8(w)
    return dequantize_symmetric(q, scale)

def quantize_per_channel_int8(w, axis=0):
    """One scale per ROW (output channel): a loud row spends bits only on itself."""
    reduce_dims = [d for d in range(w.dim()) if d != axis]      # all axes but the per-channel one
    amax = w.abs().amax(dim=reduce_dims, keepdim=True)          # per-row max magnitude
    scale = amax / INT8_QMAX                                    # one scale per row, broadcast over its columns
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(w / scale), -INT8_QMAX, INT8_QMAX)
    return q * scale

def quantize_group_wise_int8(w, group_size=GROUP_SIZE):
    """One scale per GROUP of consecutive columns: isolates a column outlier to its group."""
    rows, cols = w.shape
    assert cols % group_size == 0, "cols must divide group_size cleanly"
    wg = w.view(rows, cols // group_size, group_size)          # last axis is one group of columns
    amax = wg.abs().amax(dim=-1, keepdim=True)                 # per-group max magnitude
    scale = amax / INT8_QMAX                                   # only the group holding the outlier inflates
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(wg / scale), -INT8_QMAX, INT8_QMAX)
    return (q * scale).view(rows, cols)

rows, cols = 8, 128
w = torch.randn(rows, cols, device=DEVICE) * 0.1              # tame weights ~ N(0, 0.1)
err_pt_clean = reconstruction_error(w, quantize_per_tensor_int8(w))
outlier_col = 17
w_out = w.clone()
w_out[:, outlier_col] = torch.randn(rows, device=DEVICE) * 10.0   # one ~100x-louder column, in every row
err_pt = reconstruction_error(w_out, quantize_per_tensor_int8(w_out))
err_pc = reconstruction_error(w_out, quantize_per_channel_int8(w_out))
err_pg = reconstruction_error(w_out, quantize_group_wise_int8(w_out, 16))
print(f"clean matrix ({rows}x{cols}, ~N(0,0.1)): per-tensor mean|err| = {err_pt_clean:.6f}")
print(f"\nwith ONE 100x-outlier COLUMN (input channel {outlier_col}, in every row):")
print(f"  per-tensor  mean|err| : {err_pt:.6f}   <- loud column sets ONE global scale")
print(f"  per-channel mean|err| : {err_pc:.6f}   <- per-ROW scale does NOT help")
print(f"  per-group   mean|err| : {err_pg:.6f}   <- per-GROUP isolates the outlier")
print(f"  per-tensor blew up {err_pt/err_pt_clean:.0f}x; per-group is {err_pt/err_pg:.0f}x better than per-tensor")
assert err_pt / err_pt_clean > 5.0, "the column outlier must inflate per-tensor error"
assert err_pg < err_pt, "per-group must beat per-tensor under a column outlier"
assert err_pg < err_pc, "per-group (columns) must beat per-row under a COLUMN outlier"

clean matrix (8x128, ~N(0,0.1)): per-tensor mean|err| = 0.000807

with ONE 100x-outlier COLUMN (input channel 17, in every row):
  per-tensor  mean|err| : 0.031840   <- loud column sets ONE global scale
  per-channel mean|err| : 0.016886   <- per-ROW scale does NOT help
  per-group   mean|err| : 0.002333   <- per-GROUP isolates the outlier
  per-tensor blew up 39x; per-group is 14x better than per-tensor


## 3. Int4 group-wise — memory vs error

Production 4-bit (GPTQ/AWQ/GGUF) stores weights group-wise: split each row into groups of `group_size`, give each group its own scale. Each weight costs `4/8 = 0.5` byte for the code plus `2/group_size` bytes for the shared FP16 scale. We quantize a real weight block to int4 and compare its error and bytes/param against int8.

In [5]:
def quantize_group_wise_int4(w, group_size=GROUP_SIZE):
    """Quantize to int4 with one scale per group of `group_size` consecutive weights."""
    rows, cols = w.shape
    assert cols % group_size == 0, "cols must divide group_size cleanly"
    wg = w.view(rows, cols // group_size, group_size)
    amax = wg.abs().amax(dim=-1, keepdim=True)               # per-group max magnitude
    scale = amax / INT4_QMAX                                 # one scale per group, on the 4-bit grid [-7,7]
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(wg / scale), -INT4_QMAX, INT4_QMAX)
    return (q * scale).view(rows, cols)

def int4_bytes_per_param(group_size=GROUP_SIZE):
    """0.5 byte for the 4-bit code + one fp16 scale (2 bytes) amortized over `group_size` weights."""
    return INT4_BITS / INT8_BITS + FP16_BYTES_PER_PARAM / group_size

w4 = torch.randn(256, 1024, device=DEVICE) * 0.1
err_int4 = reconstruction_error(w4, quantize_group_wise_int4(w4, GROUP_SIZE))
err_int8 = reconstruction_error(w4, quantize_per_channel_int8(w4))
b_fp16, b_int8, b_int4 = FP16_BYTES_PER_PARAM, 1.0, int4_bytes_per_param(GROUP_SIZE)
print(f"weight block: {tuple(w4.shape)}, group_size = {GROUP_SIZE}")
print(f"  bytes/param : fp16 = {b_fp16:.3f}   int8 = {b_int8:.3f}   int4 = {b_int4:.4f} (0.5 code + {FP16_BYTES_PER_PARAM/GROUP_SIZE:.4f} scale)")
print(f"  int4 mean|err|: {err_int4:.6f}   int8 per-channel mean|err|: {err_int8:.6f}")
print(f"  int4 vs fp16 memory: {b_int4/b_fp16:.3f}x  ({(1 - b_int4/b_fp16)*100:.1f}% smaller)")
assert b_int4 < b_int8 < b_fp16, "int4 must use fewer bytes/param than int8 than fp16"
assert err_int4 > err_int8, "int4 must have larger error than int8 (fewer bits)"

weight block: (256, 1024), group_size = 64
  bytes/param : fp16 = 2.000   int8 = 1.000   int4 = 0.5312 (0.5 code + 0.0312 scale)
  int4 mean|err|: 0.009132   int8 per-channel mean|err|: 0.000679
  int4 vs fp16 memory: 0.266x  (73.4% smaller)


## 4. The 70B headline

Putting bytes/param against a real parameter count gives the number interviewers ask for: Llama-2-70B at fp16 (two 80 GB GPUs) vs int4 (one 40 GB GPU).

In [6]:
def model_memory_gb(num_params, bytes_per_param):
    """Weight memory in GB (base-10, the convention model cards use)."""
    return num_params * bytes_per_param / 1e9

params_70b = 70e9
for name, bpp in (("fp16", b_fp16), ("int8", b_int8), ("int4 (group)", b_int4)):
    print(f"  70B @ {name:<12}: {model_memory_gb(params_70b, bpp):6.1f} GB")
gb_fp16, gb_int4 = model_memory_gb(params_70b, b_fp16), model_memory_gb(params_70b, b_int4)
print(f"  -> int4 fits 70B in {gb_int4:.0f} GB (from {gb_fp16:.0f} GB) — one 40GB GPU instead of two 80GB")
assert gb_int4 < 40.0 < gb_fp16, "int4 must drop 70B under a single-GPU budget"
print("\nall asserts passed.")

  70B @ fp16        :  140.0 GB
  70B @ int8        :   70.0 GB
  70B @ int4 (group):   37.2 GB
  -> int4 fits 70B in 37 GB (from 140 GB) — one 40GB GPU instead of two 80GB

all asserts passed.


## Takeaways

- Quantization is one formula: `q = round(x/s) + z`, `x_hat = s*(q - z)`, error `<= s/2`. Adding a bit halves the step and the error.
- **Symmetric** (z=0) for weights; **asymmetric** for lopsided activations — sliding the ruler avoids wasting codes.
- The **outlier problem** is the crux: one loud channel forces a huge scale. Per-tensor breaks (39× here); per-row can't help a *column* outlier; **per-group** fixes it. Every production method (LLM.int8(), GPTQ, AWQ, SmoothQuant, NF4, GGUF) is a different answer to outliers.
- **int4 group-wise** is ~0.53 bytes/param (73% smaller than fp16) and nearly lossless on weights — 70B drops from 140 GB to ~37 GB.

See the [Quantization concept page](../10-Quantization.md) for the full derivation, diagrams, and the method-by-method walkthrough, and [`make_figures.py`](make_figures.py) for the figures.